# **🏦 Modelo de Riesgo Crediticio - Predicción de Incumplimiento del Crédito con Redes Neuronales 💵**

### Descripción del problema
El riesgo crediticio es uno de los mayores desafíos en la industria financiera. Esto debido a que entidades bancarias requieren evaluar la probabilidad de que un cliente **incumpla con el pago de su crédito** antes de concederlo.

En este proyecto se construye un **modelo de red neuronal**
para predecir la probabilidad de incumplimiento (*default*) a partir de características del cliente y su historial crediticio.


### Objetivo
El objetivo de la actividad es estimar la probabilidad de que un cliente sea **"mal pagador"** (valor 1)
usando el dataset *Credit Risk Dataset* de Kaggle, esto se logrará al optimizar la arquitectura de la red neuronal y representar el resultado como una **scorecard**.


### Variable objetivo
La variable `loan_status` es transformada en una variable binaria:
- **0** → Buen pagador (pagó su crédito completamente)
- **1** → Mal pagador (no pagó su crédito completamente)
- **NA** → Casos excluidos del entrenamiento (no se tiene certeza sobre si el cliente es buen o mal pagador).

***IMPORTANTE: Si usted va a ejecutar los códigos de este Notebook se recomienda cambiar el tipo de entorno de ejecución GPU T4***

## **1. Importación de las librerías**

Iniciamos importando las librerías que necesitaremos para el desarrollo del trabajo.

In [ ]:
#1 --- Importación de la liberías requeridas
!pip install pandas numpy matplotlib seaborn scikit-learn tensorflow imbalanced-learn --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, ConfusionMatrixDisplay, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from imblearn.over_sampling import SMOTE
import random
import os

#Además fijamos las semillas para que los códigos sean reproducibles y obtenga el mismo resultado en cada ejecución
SEED = 42
random.seed(SEED) # Python
np.random.seed(SEED)  # NumPy
tf.random.set_seed(SEED)  # TensorFlow/Keras
os.environ['PYTHONHASHSEED'] = str(SEED) # Hashing de Python

#Con esto hacemos el entrenamiento más determinístico, aunque se sacrifique capacidad de cómputo
tf.config.experimental.enable_op_determinism()

## **2. Carga de datos y exploración del Dataset**

Para iniciar con el trabajo, cargaremos los datos. Para este paso, hay que tener en cuenta que se debe descargar el archivo "loan.csv" del repositorio en GitHub y posteriormente se debe subirlo en este Notebook de Google Colab para que los siguientes códigos se ejecuten correctamente y se pueda realizar la actividad.

In [ ]:
#2 --- Carga de datos y exploración inicial del Dataset

#El parámetro low_memory es para evitar que se asignen
#tipos de datos mezclados a las columnas 19 y 55 del Dataframe
df = pd.read_csv('loan.csv',  low_memory=False)

# Exploración inicial del Dataset
print("Tamaño del Dataset:", df.shape)
print("\nPrimeras filas:")
print(df.head())
print("\nTipos de datos:")
print(df.dtypes)
print("\nEstadísticas descriptivas:")
print(df.describe())
print("\nValores únicos de loan_status:")
print(df['loan_status'].value_counts())
print("\nValores nulos por columna:")
print(df.isnull().sum())

FileNotFoundError: [Errno 2] No such file or directory: 'loan.csv'

A partir de la exploración del Dataset podemos analizar que es un Dataset de gran tamaño que contiene **887,379 filas y 74 columnas**. Además, se puede deducir que tras eliminar los valores NA de la variable objetivo **'loan_status'**, asociados a los valores: **Current**, **In Grace Period**, **Issued** y **Late (16-30 days)**, nos quedarán aproximadamente 268,530 registros para modelar. De estos valores que nos quedan los malos pagadores son acerca del **21.9%** de los registros **(asociados a los valores: Charged Off, Does not meet the credit policy. Status:Charged Off, Late (31-120 days) y Default)**, mientras que los buenos pagadores corresponden al **78.1%** de los registros aproximadamente **(asociados a los valores: Fully Paid, Does not meet the credit policy. Status:Fully Paid)**. A partir de lo anterior se puede concluir que la base de datos está desbalanceada, lo cual se debe tratar analizando no únicamente la cantidad de personas mal clasificadas, sino usando otro tipo de técnicas como f1 y recall. También a partir del análisis exploratorio se puede observar que existen varias columnas con todos o casi todos los elementos marcados con valores nulos y que no aportan nada en realidad al entrenamiento del modelo y que por lo tanto se deben excluir, y a su vez existen otras columnas que cargan valores importantes para ser considerados al elaborar la red neuronal, como **annual_inc**, **int_rate**, **dti**, **delinq_2yrs** y **loan_amnt**.


## **3. Preprocesamiento y limpieza de los datos**

Empezaremos por definir nuestra variable objetivo y mapear sus valores de una manera que nos sea útil para la creación del modelo.

In [ ]:
#3.1 --- Definición de la variable objetivo

# Mapeamos la variable loan_status de String a binaria según nuestras necesidades (1:mal pagador, 0:buen pagador)
mapa_de_estado = {
    'Fully Paid': 0,
    'Does not meet the credit policy. Status:Fully Paid': 0,
    'Charged Off': 1,
    'Late (31-120 days)': 1,
    'Default': 1,
    'Does not meet the credit policy. Status:Charged Off': 1,
    'Current': np.nan,
    'Issued': np.nan,
    'In Grace Period': np.nan,
    'Late (16-30 days)': np.nan,
}

#En la columna 'target' almacenamos nuestra variable objetivo
df['target'] = df['loan_status'].map(mapa_de_estado)

# Eliminamos los NA
df_modelo = df[df['target'].notna()].copy()
#Cambiamos el tipo de dato de la variable objetivo a entero
df_modelo['target'] = df_modelo['target'].astype(int)

print("Distribución de la variable objetivo:")
print(df_modelo['target'].value_counts())
#El promedio nos sirve para observar el porcentaje de malos pagadores
print(f"\nProporción de malos pagadores: {df_modelo['target'].mean():.2%}")
print(f"Proporción de buenos pagadores: {1-df_modelo['target'].mean():.2%}")
print(f"\nRegistros para modelar: {df_modelo.shape[0]:,}")

Ya con la variable objetivo definida se puede confirmar nuestra hipótesis sobre el desbalance de los datos y la cantidad de registros que tendremos para entrenar y validar el modelo de red neuronal.

Ahora procedemos a eliminar las columnas inútiles (que contienen muchos valores nulos y que no aportan información relevante para el modelo).

In [ ]:
#3.2 --- Eliminación de columnas con demasiados nulos o con poco valor para la predicción de valores

# Definimos nuestro umbral para eliminar las columnas con 80%+ de nulos
umbral = 0.8
nulos = df_modelo.isnull().mean()
#Tomamos los nombres de las columnas a eliminar de nuestro Dataframe por pasarse del umbral
cols_drop_nulos = nulos[nulos >= umbral].index.tolist()

#Se eliminarán a su vez aquellas columnas identificadoras que no aportan información relevante para la predicción
#del modelo, incluyendo 'loan_status' que ya no la necesitamos pues incluímos la columna 'target' como variable objetivo
cols_drop_id = ['id', 'member_id', 'url', 'desc', 'title',
                'zip_code', 'addr_state', 'loan_status',
                'emp_title', 'earliest_cr_line', 'last_pymnt_d',
                'next_pymnt_d', 'last_credit_pull_d', 'issue_d']

#Fusionamos ambos criterios en una lista de columnas a eliminar
cols_to_drop = list(set(cols_drop_nulos + cols_drop_id))

#Y finalmente las borramos del Dataset para nuestro modelo
df_modelo = df_modelo.drop(columns=[c for c in cols_to_drop if c in df_modelo.columns])

#Verificamos la cantidad de columnas eliminadas y restantes del Dataset, junto con el cambio que esta trae en su tamaño
print(f"Columnas eliminadas por nulos: {len(cols_drop_nulos)}")
print(f"Columnas eliminadas por ser identificadores: {len(cols_drop_id)}")
print(f"Columnas restantes: {df_modelo.shape[1]}")
print(f"Tamaño final del Dataset: {df_modelo.shape}")

Ahora, haremos la separación de variables numéricas y categóricas, para que el modelo sepa como comportarse según el tipo de dato de cada columna.

In [ ]:
#3.3 --- Separación de variables numéricas y categóricas

# Excluimos la variable objetivo
caracteristicas = [c for c in df_modelo.columns if c != 'target']

# Separamos las columnas por tipo de dato
num_cols = df_modelo[caracteristicas].select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = df_modelo[caracteristicas].select_dtypes(include=['object']).columns.tolist()

print(f"Variables numéricas ({len(num_cols)}):")
print(num_cols)
print()
print(f"Variables categóricas ({len(cat_cols)}):")
print(cat_cols)

En total podemos observar que hay 41 variables sin contar la variable objetivo 'target', lo cual corresponde con las dimensiones exploradas en la celda anterior.

Ahora, para tener datos más limpios, debemos buscar alguna manera de implantar ciertos valores "justos" para los valores nulos que pueden afectar negativamente nuestro modelo o considerar eliminarlos. Para nuestro caso, y en búsqueda de mantener una gran cantidad de registros disponibles sin reducirlos abruptamente, hemos optado por implantar ciertos valores específicos para los valores nulos que se tengan en el Dataframe según la naturaleza del tipo de dato presente en cada columna. Para los datos numéricos usaremos la mediana de los datos de la columna, mientras que para los datos categóricos, elegiremos la moda.

In [ ]:
#3.4 --- Imputación de valores nulos

# Para las variables numéricas → mediana (la elegimos pues es una opción robusta frente a los outliers)
for col in num_cols:
    mediana = df_modelo[col].median()
    df_modelo[col] = df_modelo[col].fillna(mediana)

#Para las variables categóricas → moda
for col in cat_cols:
    moda = df_modelo[col].mode()[0] #Si existen varias modas, se toma la primera (la menor)
    df_modelo[col] = df_modelo[col].fillna(moda)

print(f"cantidad de valores nulos restantes: {df_modelo.isnull().sum().sum()}") #Doble suma por tener dos dimensiones el Dataframe

Ahora nuestro Dataframe está libre de valores nulos y está casi listo para ser utilizado, únicamente falta codificar las variable categóricas para que la red pueda tomar valores numéricos de todas las columnas y se puede entrenar correctamente. Así que para esto usaremos dos tipos de codificación: **One Hot Encoding** y **mapeo manual con diccionario**, esto tiene un sentido, y es que en mapeo manual el orden en que se codifican las variables es relevante, mientras que en One Hot Encoding no. Y esto es útil según el tipo de variable categórica, para nuestro caso, las variables **'grade'** y **'sub_grade'** son categóricas ordinales, por lo que es mejor usar mapeo manual con diccionario para estas dos, mientras que el resto de variables categóricas **(home_ownership, purpose, verification_status, term, emp_length, pymnt_plan, initial_list_status y application_type)** son nominales, lo cual las hace más propensas a utilizar One Hot Encoding para su codificación evitando que se le asigne un orden que no es útil en el contexto.

In [ ]:
#3.5.1 --- Encoding de variables categóricas ordinales

# Mapeo manual para la variable grade (orden real: A=mejor, G=peor)
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
if 'grade' in df_modelo.columns:
    df_modelo['grade'] = df_modelo['grade'].map(grade_map)
    print("La variable grade ha sido codificada con orden natural")

# Mapeo manual para la variable sub_grade (A1=mejor, G5=peor)
if 'sub_grade' in df_modelo.columns:
    # Generamos el orden: A1,A2,A3,A4,A5,B1,B2,...,G5
    subgrades = ["A1","A2","A3","A4","A5",
                 "B1","B2","B3","B4","B5",
                 "C1","C2","C3","C4","C5",
                 "D1","D2","D3","D4","D5",
                 "E1","E2","E3","E4","E5"]
    subgrade_map = {sg: i+1 for i, sg in enumerate(subgrades)}
    df_modelo['sub_grade'] = df_modelo['sub_grade'].map(subgrade_map)
    print("La variable sub_grade ha sido codificada con orden natural")

# Guardamos los encoders ordinales
ordinal_cols = ['grade', 'sub_grade']

# Implamantamos la mediana de cada columna a posibles Outliers que no se hayan
# mapeado anteriormente para evitar errores en el futuro
if 'grade' in df_modelo.columns:
    df_modelo['grade'] = df_modelo['grade'].fillna(df_modelo['grade'].median())

if 'sub_grade' in df_modelo.columns:
    df_modelo['sub_grade'] = df_modelo['sub_grade'].fillna(df_modelo['sub_grade'].median())

In [ ]:
#3.5.2 --- Encoding de variables categóricas nominales

# Variables a las que aplicaremos One Hot Encoding
nominal_cols = [col for col in cat_cols
                if col not in ordinal_cols
                and col in df_modelo.columns]

print(f"Variables con One Hot Encoding: {nominal_cols}")

# Aplicamos One Hot Encoding
df_modelo = pd.get_dummies(df_modelo, columns=nominal_cols, drop_first=True)

# Haremos el pequeño ajuste de convertir columnas booleanas a enteras y rellenar los NaN restantes
bool_cols = df_modelo.select_dtypes(include='bool').columns
df_modelo[bool_cols] = df_modelo[bool_cols].astype(int)

for col in df_modelo.columns:
    #Si hay algún valor nulo se cambia por la mediana si es numérico, sino por 0
    if df_modelo[col].isna().any():
        if df_modelo[col].dtype in ['float64', 'int64']:
            df_modelo[col] = df_modelo[col].fillna(df_modelo[col].median())
        else:
            df_modelo[col] = df_modelo[col].fillna(0)


print()
print(f"One Hot Encoding completado")
print(f"Tamaño después del encoding: {df_modelo.shape}")
print(f"\nColumnas del dataset:")
print(df_modelo.columns.tolist())

Tras aplicar el One Hot Encoding la cantidad de columnas que se tiene ha aumentado producto de la manera en la que esta forma de codificación hace su trabajo. Ya con esto estamos listos para comenzar con el desarrollo del modelo, los datos ya han sido preprocesados de manera adecuada.

## **4. Análisis descriptivo y elaboración de hipótesis**

Para este siguiente paso, buscamos entender los datos antes de generar el modelo, para que cuando lo realicemos este no carezca de sentido lógico o estructural, lo cual nos puede brindar mejores resultados al tener mayor dominio del problema. De igual manera se podrán plantear hipótesis sobre que variables se consideran más importantes o menos importantes a la hora de predecir si un determinado cliente cumplirá o inclumplirá con el pago de su crédito.

Para lograr esto, nos enfocaremos en observar que tan desbalanceado está el Dataset entre buenos y malos pagadores para entender qué técnicas especiales necesitaremos para entrenar el modelo, además de comparar como se distribuyen variables numéricas entre buenos y malos pagadores para comprender que variables pueden ser más útiles para el modelo, lo mismo con las variables categóricas. Y finalmente, para comprender cuales son las correlaciones entre las variables y con el target, la cuales pueden favorecer o sabotear el proceso de aprendizaje del modelo.

Empezaremos observando la distribución de los valores del target.

In [ ]:
#4.1 --- Análisis descriptivo - Distribución del target

# Conteos y proporciones
conteos = df_modelo['target'].value_counts()
proporciones = df_modelo['target'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico 1 — Conteo absoluto
axes[0].bar(['Buen pagador (0)', 'Mal pagador (1)'],
            conteos.values,
            color=['#2ecc71', '#e74c3c'],
            edgecolor='black', width=0.5)
axes[0].set_title('Distribución absoluta del target', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Número de registros')
for i, v in enumerate(conteos.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

# Gráfico 2 — Proporción
axes[1].pie(proporciones.values,
            labels=['Buen pagador (0)', 'Mal pagador (1)'],
            colors=['#2ecc71', '#e74c3c'],
            autopct='%1.1f%%',
            startangle=90,
            wedgeprops={'edgecolor': 'black'})
axes[1].set_title('Proporción del target', fontsize=13, fontweight='bold')

plt.suptitle('Figura 1. Distribución de la variable objetivo',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig1_distribucion_target.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBuenos pagadores:  {conteos[0]:,} ({proporciones[0]:.1f}%)")
print(f"Malos pagadores:   {conteos[1]:,} ({proporciones[1]:.1f}%)")
print(f"Ratio desbalance:  {conteos[0]/conteos[1]:.1f}:1")

A partir de la gráfica, podemos analizar que hay aproximadamente **3.6** veces más buenos pagadores que malos pagadores y que del conjunto de datos disponible **209,711** son buenos pagadores, lo cual equivale a un **78.1%** del total de los registros, y que a su vez, hay **58,819** malos pagadores, lo cual equivale al **21.9%** del total. Lo cual nos confirma claramente que los datos están desbalanceados y que debemos tomar medidas con respecto a eso.

Ahora pasaremos a analizar las variables numéricas vs el target.

Compararemos cómo se distribuyen las variables numéricas más importantes entre buenos y malos pagadores usando boxplots. Si una variable tiene distribuciones diferentes entre las dos clases, es una buena señal de que será útil para la elaboración del modelo.

Las variables que analizaremos son:
*   **int_rate — tasa de interés**
*   **dti — relación deuda/ingreso**
*   **annual_inc — ingreso anual**
*   **loan_amnt — monto del crédito**
*   **installment — cuota mensual**
*   **revol_util — utilización del crédito rotativo**


In [ ]:
#4.2 --- Análisis descriptivo - Variables numéricas vs target

# Variables numéricas a analizar
vars_numericas = ['int_rate', 'dti', 'annual_inc', 'loan_amnt', 'installment', 'revol_util']

# Necesitamos el target sin codificar para graficar
# Creamos una columna de etiquetas legibles
df_plot = df_modelo.copy()
df_plot['target_label'] = df_plot['target'].map({0: 'Buen pagador', 1: 'Mal pagador'})

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, var in enumerate(vars_numericas):
    if var in df_plot.columns:
        sns.boxplot(data=df_plot, x='target_label', y=var, hue='target_label',
                    palette={'Buen pagador': '#2ecc71', 'Mal pagador': '#e74c3c'},
                    legend=False, ax=axes[i], width=0.5)
        axes[i].set_title(f'{var}', fontsize=12, fontweight='bold')
        axes[i].set_xlabel('')
        axes[i].set_ylabel(var)

        # Añadimos la media de cada grupo
        medias = df_plot.groupby('target_label')[var].mean()
        for j, (label, media) in enumerate(medias.items()):
            axes[i].text(j, axes[i].get_ylim()[1] * 0.95,
                        f'Media: {media:.1f}',
                        ha='center', fontsize=9,
                        color='black', fontweight='bold')

plt.suptitle('Figura 2. Distribución de variables numéricas por tipo de pagador',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_numericas_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla de medias por grupo
print("\nMedias por grupo:")
print(df_plot.groupby('target_label')[vars_numericas].mean().round(2).T)

Con base en lo anterior, se puede observar que **int_rate** es la señal más clara de riesgo, ya que las cajas del gráfico correspondiente a esta variable para buenos y malos pagadores es la que más separada está la una de la otra, lo cual suele generar mayor confianza en la predicción de un cliente, lo que nos permite analizar que altas tasas de crédito están asociadas al incumplimiento en el pago de las mismas.

El **dti** también parece separar bien los grupos y sugiere que quienes ya tienen muchas deudas en relación con su ingreso tienen mayor probabilidad de incumplir, pero no es la mejor señal de separación.

El **annual_inc** muestra que los buenos pagadores ganan más que los malos pagadores pero no es la mejor variable para decidir porque sus medias están muy cercanas y no separa a ambos grupos con claridad.

El **loan_amnt** tampoco sirve mucho para el modelo, pero señala que los malos pagadores piden más en sus créditos que los buenos pagadores.

El **installment** tiene un índice y comportamiento muy similar a la variable anterior lo cual puede indicar correlación entre ambas variables.

El **revol_util** señala que los malos pagadores usan más dinero del crédito disponible que quienes son buenos pagadores, aunque la diferencia no es enorme y tampoco servirá mucho como factor diferencial en el modelo.

De la misma manera analizaremos las variables categóricas vs el target, comparando cómo se distribuyen las variables categóricas más importantes entre buenos y malos pagadores usando gráficos de barras. De igual manera, si una variable tiene distribuciones diferentes entre las dos clases, es una buena señal de que será útil para el modelo.

Las variables que analizaremos son:
*   **grade — calificación crediticia del préstamo**
*   **purpose — propósito del crédito**
*   **home_ownership — tipo de vivienda del solicitante**
*   **verification_status — estado de verificación de ingresos**
*   **emp_length — antigüedad laboral**
*   **term — plazo del crédito**

In [ ]:
#4.3 --- Análisis descriptivo - Variables categóricas vs target

# Recuperamos las columnas originales antes del encoding
# Trabajaremos con df_modelo pero reconstruyendo las categóricas desde df
cat_vars_analisis = ['grade', 'purpose', 'home_ownership', 'verification_status', 'emp_length', 'term']

fig, axes = plt.subplots(3, 2, figsize=(18, 18))
axes = axes.flatten()

for i, var in enumerate(cat_vars_analisis):
    ax = axes[i]

    # Reconstruimos la variable original desde df (antes del encoding)
    temp = df[[var, 'loan_status']].copy()
    temp['target'] = df['loan_status'].map(mapa_de_estado)
    temp = temp[temp['target'].notna()]
    temp['target'] = temp['target'].astype(int)

    # Calculamos tasa de incumplimiento por categoría
    tasa = temp.groupby(var)['target'].mean().sort_values(ascending=False) * 100
    conteo = temp.groupby(var)['target'].count()

    colores = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(tasa)))

    bars = ax.bar(tasa.index, tasa.values, color=colores, edgecolor='black', linewidth=0.5)

    # Añadimos el n de cada categoría sobre cada barra
    for bar, cat in zip(bars, tasa.index):
        n = conteo[cat]
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.3,
                f'n={n:,}',
                ha='center', va='bottom', fontsize=7, rotation=45)

    ax.set_title(f'Tasa de incumplimiento por {var}', fontsize=12, fontweight='bold')
    ax.set_ylabel('% Malos pagadores')
    ax.set_xlabel(var)
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylim(0, tasa.max() * 1.25)

    # Línea de referencia: tasa global
    tasa_global = df_modelo['target'].mean() * 100
    ax.axhline(tasa_global, color='navy', linestyle='--', linewidth=1.2, label=f'Promedio: {tasa_global:.1f}%')
    ax.legend(fontsize=8)

plt.suptitle('Figura 3. Tasa de incumplimiento por variable categórica',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig3_categoricas_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

Basados en el gráfico anterior, podemos deducir que **grade** es la variable categórica con mayor poder de separación pues tiene una tendencia marcada de que mientras mayor sea el grado (más cercano a G) mayor será la probabilidad de que el cliente incumpla con el pago de su crédito.

**Purpose** muestra que los créditos para **small bussiness** son los más riesgosos y que para **wedding** y **car** son los más seguros, también aparenta ser una buena variable para separar ambos casos.

El **home_ownership** parece ser más estable por lo que es más riesgoso para nuestro modelo tomar decisiones basadas en dicha variable.

**verification_status** suele tener un comportamiento similar a la anterior variable y parece que la clasificación es contraintuitiva puesto que los clientes con ingresos **verified** presentan más incumplimiento.

**emp_length** Es la variable con menor poder discriminativo de todas debido a la similitud presentada en todos los posibles valores de la misma con respecto a la media.

Y el **term** si puede ser una variable muy útil, ya que nos permite analizar que los créditos a 60 meses presetan mayor índice de incumplimiento frente a los créditos a 32 meses, lo cual suena bastante lógico.

Finalmente estudiaremos la correlación entre las variables con el target y entre las variables numéricas.

In [ ]:
#4.4 --- Análisis descriptivo - Correlaciones con el target

# Tomamos solo las variables numéricas originales + target
vars_correlacion = num_cols + ['target']
df_corr = df_modelo[vars_correlacion].copy()

# --- Gráfico 1: Correlaciones con el target (barras horizontales) ---
correlaciones_target = df_corr.corr()['target'].drop('target').sort_values()

fig, ax = plt.subplots(figsize=(10, 12))

colores = ['#e74c3c' if v > 0 else '#2ecc71' for v in correlaciones_target.values]
bars = ax.barh(correlaciones_target.index, correlaciones_target.values,
               color=colores, edgecolor='black', linewidth=0.5)

ax.axvline(0, color='black', linewidth=1)
ax.axvline(0.05, color='gray', linewidth=0.8, linestyle='--', alpha=0.6)
ax.axvline(-0.05, color='gray', linewidth=0.8, linestyle='--', alpha=0.6)

# Etiquetas de valor
for bar, val in zip(bars, correlaciones_target.values):
    ax.text(val + (0.002 if val >= 0 else -0.002),
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}',
            va='center', ha='left' if val >= 0 else 'right', fontsize=8)

ax.set_title('Figura 4a. Correlación de variables numéricas con el target',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Correlación de Pearson')
ax.set_ylabel('Variable')
plt.tight_layout()
plt.savefig('fig4a_correlaciones_target.png', dpi=150, bbox_inches='tight')
plt.show()

print()

# --- Gráfico 2: Heatmap de correlaciones entre variables numéricas ---
# Nos quedamos solo con las variables con correlación > 0.05 o < -0.05 con el target
vars_relevantes = correlaciones_target[abs(correlaciones_target) > 0.05].index.tolist() + ['target']
df_corr_relevante = df_corr[vars_relevantes]

fig, ax = plt.subplots(figsize=(14, 10))

mask = np.triu(np.ones_like(df_corr_relevante.corr(), dtype=bool))

sns.heatmap(df_corr_relevante.corr(),
            mask=mask,
            annot=True, fmt='.2f',
            cmap='RdYlGn_r',
            center=0,
            linewidths=0.5,
            ax=ax,
            annot_kws={'size': 8})

ax.set_title('Figura 4b. Heatmap de correlaciones entre variables relevantes',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4b_heatmap_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Resumen numérico ---
print("\nTop 10 variables más correlacionadas con el target:")
print(correlaciones_target.abs().sort_values(ascending=False).head(10).to_string())

Concluímos que las variables con correlación positiva más fuerte con el target inducen más riesgo, como lo son: **recoveries (0.390), out_prncp y out_prncp_inv (0.350) y collection_recovery_fee (0.267)**. Pero eso es porque estas variables son consecuencia del incumplimiento más no la causa, por lo que se deben tratar con cuidado. Fuera de estas, **int_rate (0.255)** es la variable más útil del grupo porque es una causa real del incumplimiento, no una consecuencia. La variable **dti (0.134) y revol_util (0.100)** también muestran correlación positiva moderada. Por contraparte Las variables con correlación negativa fuerte con el target generan más confianza en el pago del crédito, como ocurre con **total_rec_prncp (-0.487)** que es la variable más correlacionada de todas, pero sigue derivandose de la consecuencia de haber pagado, al igual que con **total_pymnt (-0.385), total_pymnt_inv (-0.382) y last_pymnt_amnt (-0.412)**. Mientras que una variable útil puede ser **annual_inc (-0.057)**.

El mapa de calor entre variables numéricas muestra grupos de variables altamente correlacionadas entre sí que es importante tener en cuenta. **loan_amnt, funded_amnt y funded_amnt_inv** tienen correlaciones de **0.99 y 1.00** entre ellas. **total_pymnt y total_pymnt_inv** tienen correlación de **0.99**. **out_prncp y out_prncp_inv** tienen correlación de **1.00**. Tener estos grupos de variables dentro del Dataset puede generar redundacia innecesaria.

## **5. Modelo base implementando regresión logística**

Comenzaremos construyendo un modelo de red neuronal que funcionará como base, para ello, implementaremos una regresión logística y observaremos los resultados para posteriormente optimizarla.

Para ello iniciaremos seleccionando las features o características que utilizará el modelo, por lo que excluiremos aquellas columnas que estén duplicadas (que generen información redundante) o que contengan data leakage, es decir, aquellas variables que contienen fuga de datos debido a que en realidad son consecuencia del target y no la causa del mismo.

In [ ]:
#5 --- Modelo base: Regresión Logística

#5.1 --- Selección de features
# Excluimos variables con data leakage y duplicadas identificadas en el análisis anterior
vars_data_leakage = [
    'recoveries', 'collection_recovery_fee', 'total_rec_prncp',
    'total_pymnt', 'total_pymnt_inv', 'total_rec_int', 'total_rec_late_fee',
    'last_pymnt_amnt', 'out_prncp', 'out_prncp_inv'
]

# Excluimos duplicadas (nos quedamos con loan_amnt y descartamos funded_amnt y funded_amnt_inv)
vars_duplicadas = ['funded_amnt', 'funded_amnt_inv']

vars_excluir = vars_data_leakage + vars_duplicadas

feature_cols = [c for c in df_modelo.columns
                if c != 'target' and c not in vars_excluir]

#Definimos nuestro Dataset de entradas y nuestra Serie de salidas
X = df_modelo[feature_cols]
y = df_modelo['target']

print(f"Características utilizadas: {len(feature_cols)}")
print(f"Registros totales: {X.shape[0]:,}")

Elegiremos aquellas variables más representativas para el modelo, con la idea de no saturarlo de parámetros y hacerlo más accesible a consultar y clasificar nuevos clientes según los valores que se ingrese sobre estos.

In [ ]:
#5.2 --- Selección manual de variables

#Elegimos las variables más representativas que son causa del resultado
variables_seleccionadas = [
    'int_rate',
    'dti',
    'annual_inc',
    'loan_amnt',
    'revol_util',
    'delinq_2yrs',
    'pub_rec',
    'open_acc',
    'total_acc',
    'grade',
    'term_ 60 months',
    'home_ownership_RENT',
    'home_ownership_MORTGAGE',
    'home_ownership_OWN',
    'purpose_small_business',
    'purpose_debt_consolidation',
    'purpose_credit_card',
    'purpose_home_improvement',
    'purpose_medical',
    'purpose_other',
]

# Filtramos X para quedarnos solo con las variables seleccionadas
# Verificamos primero que todas existan en el dataset
vars_disponibles = [v for v in variables_seleccionadas if v in X.columns]

X = X[vars_disponibles]

print(f"Variables utilizadas: {len(vars_disponibles)}")
print(f"Variables: {vars_disponibles}")
print(f"Tamaño de X: {X.shape}")

Ahora con los datos de entrada y salida esperada separados, vamos a dividir los conjuntos de datos que servirán para el entrenamiento y para validación del modelo. Usaremos el **80%** de los datos para entrenamiento y el **20%** para validación.

In [ ]:
#5.3 --- Split train/test

#Con el parámetro stratify se busca que la proporción de malos y buenos pagadores no se altere con respecto al Dataframe original
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Tamaño del conjunto de entrenamiento: {X_train.shape[0]:,}")
print(f"Tamaño del conjunto de validación:  {X_test.shape[0]:,}")
print(f"Proporción de malos pagadores en el conjunto de entrenamiento: {y_train.mean():.2%}")
print(f"Proporción de malos pagadores el el conjunto de validación:  {y_test.mean():.2%}")
print(f"Proporción de buenos pagadores en el conjunto de entrenamiento: {1-y_train.mean():.2%}")
print(f"Proporción de buenos pagadores el el conjunto de validación:  {1-y_test.mean():.2%}")

Ya con los datos separados, entrenaremos al modelo usando la regresión logística, con un máximo de **1000** iteraciones, con el solver **lbfgs** y balanceando las clases, de manera que se penalice más una predicción incorrecta de la clase minoritaria (en este caso los malos pagadores), evitando así que siempre se prediga a un cliente como buen pagador.

In [ ]:
#5.4 --- Entrenamiento
# Usamos class_weight='balanced' para compensar el desbalance
logit = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
    solver='lbfgs'
)

#Normalizamos los datos para estabilizar el comportamiento de la red durante el entrenamiento
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

#Entrenamos el modelo con los datos de entrenamiento
logit.fit(X_train_scaled, y_train)

#Actualizamos el tamaño de entrada
input_dim = X_train_scaled.shape[1]
print(f"input_dim actualizado: {input_dim}")

print("Entrenamiento del modelo completado")

Ya con la red entrenada, evaluaremos sus resultados con el conjunto de datos de testing y generaremos un reporte de clasificación para el modelo con regresión logística, además hallaremos el valor AUC-ROC del modelo para observar su desempeño general.

In [ ]:
#5.5 --- Evaluación
y_pred = logit.predict(X_test_scaled) #Clase exacta en la que fue clasificado el cliente por el modelo
y_pred_proba = logit.predict_proba(X_test_scaled)[:, 1] #Probabilidad de ser clasificado en cada clase

#Se encuentra el valor AUC-ROC
auc = roc_auc_score(y_test, y_pred_proba)

print("\nReporte de clasificación (Regresión Logística)\n")
print(classification_report(y_test, y_pred,
                             target_names=['Buen pagador', 'Mal pagador']))
print(f"AUC-ROC: {auc:.4f}")

En el reporte generado se pueden apreciar diferentes métricas y el porcentaje de acierto a cada clasificación de los clientes separados en buenos y malos pagadores. Cada métrica nos sirve para evaluar diferentes aspectos del modelo, por ejemplo, la AUC-ROC es el valor más importante y nos ayuda a determinar que el modelo tiene un **70%** de probabilidad de determinar correctamente a un cliente en el grupo al que pertenece. El **accuracy** no es muy bueno en el modelo, ya que es sólo del **65%** aunque no es la mejor medida para nuestro caso por el desbalance de los datos, de hecho si el modelo siempre predijera a todos como buenos pagadores obetendría un **78.1% de accuracy**. EL **recall** nos indica que el modelo detecta satisfactoriamente al **65%** de los malos pagadores reales, lo que también es una métrica que nos interesa mejorar dentro del modelo. La **precision** es muy baja, sólo del **34%** y nos indica la cantidad de aciertos de malos pagadores dentro del modelo cuando este los determina de esta manera, es decir la cantidad de falsos positivos, lo cual podemos aceptar en nuestro modelo si nuestro objetivo es evitar a toda costa los malos pagadores a cambio de sacrificar cierta cantidad de buenos pagadores.

In [ ]:
#5.6 --- Gráficos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Buen pagador', 'Mal pagador'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Figura 5a. Matriz de confusión\nRegresión Logística',
                  fontsize=12, fontweight='bold')

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'ROC (AUC = {auc:.4f})')
axes[1].plot([0,1], [0,1], 'k--', lw=1, label='Clasificador aleatorio')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].set_title('Figura 5b. Curva ROC\nRegresión Logística',
                  fontsize=12, fontweight='bold')
axes[1].legend(loc='lower right')

plt.suptitle('Figura 5. Evaluación del modelo base (Regresión Logística)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig5_modelo_base.png', dpi=150, bbox_inches='tight')
plt.show()

En general el modelo no es el mejor, puesto que presenta una gran cantidad de falsos positivos y muchos falsos negativos, aunque en realidad es un buen clasificador puesto que clasifica mucho mejor a los clientes en comparación con uno aleatorio.

Estos resultados sugieren que la elaboración de un modelo más complejo como una red neuronal podría entender mejor las relaciones no lineales entre las variables, lo cual podría generar mejores resultados en la clasificación de los clientes.

## **6. Elaboración de la red neuronal**

Ahora conociendo los fallos del modelo anterior, diseñemos una red neuronal para que pueda clasificar a los usuarios que solicitan un crédito con una mejor precisión.

Para empezar preparamos los datos con un factor de peso que nos ayudará a balancear el modelo.

In [ ]:
#6 --- Red Neuronal

#6.1 --- Preparación de datos
# Reutilizamos X_train_scaled, X_test_scaled, y_train, y_test del modelo base

# Calculamos el class_weight para compensar el desbalance
total = len(y_train)
buenos = (y_train == 0).sum()
malos = (y_train == 1).sum()

class_weight = {
    0: total / (2 * buenos),
    1: total / (2 * malos)
}

print(f"Estos son los class weights:")
print(f"  Buen pagador (0): {class_weight[0]:.3f}")
print(f"  Mal pagador  (1): {class_weight[1]:.3f}")

Para compensar el desbalance, los malos pagadores tienen un peso mayor que el de los buenos pagadores, según el cálculo anterior.

Ahora diseñaremos la arquitectura de la red, para esto usaremos un modelo secuencial con **2 capas ocultas** que contienen **64** y **32** neuronas, una capa de entrada con **128 neuronas** y una de salida con una sola que contendrá un valor entre **0** y **1** según la clasificación final del modelo. Usaremos una **tasa de aprendizaje de 0.001** para que el paso que dé el optimizador no sea ni muy grande para no llegar a un mínimo local, ni muy pequeño para evitar que colapse la memoria. También usaremos un **ratio de dropout del 30%** para que el modelo no caiga en el sobreentrenamiento **(overfitting)**, de esta manera cortando conexiones e impidiendo que el algoritmo "Memorice" los datos del conjunto de entrenamiento. Usaremos funciones de activación **ReLU** para la capa de entrada y las capas ocultas, mientras que para la capa de salida utilizaremos la **Sigmoide** para obtener la probabilidad de que los datos sean clasificados en uno u otro grupo como indicador de la certeza que tiene el modelo al clasificar a un determinado cliente. Las capas serán **densas**, por lo que todas las neuronas de una capa estarán conectadas con todas las neuronas de la capa siguiente. También normalizaremos las activaciones entre capas durante el entrenamiento para estabilizar y acelerar la convergencia del modelo durante el proceso de aprendizaje. Finalmente compilaremos el modelo utilizando el optimizador **Adam**, con función de pérdida **binary_crossentropy**, debido a que la variable de decisión **(target)** es binaria, y para los resultados usaremos como métricas el valor **AUC** y el **accuracy**.

In [ ]:
#6.2 --- Arquitectura de la red

#Función para crear la arquitectura de la red neuronal
def build_model(input_dim, learning_rate=0.01,
                dropout_rate=0.2, layers=[64, 32]):

    model = Sequential()

    # Capa de entrada
    model.add(Input(shape=(input_dim,)))
    model.add(Dense(layers[0], activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))

    # Capas ocultas
    for units in layers[1:]:
        model.add(Dense(units, activation='relu'))
        model.add(BatchNormalization())
        model.add(Dropout(dropout_rate))

    # Capa de salida
    model.add(Dense(1, activation='sigmoid'))

    #Compilamos el modelo
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['AUC', 'accuracy']
    )

    return model

#Construimos el modelo
model = build_model(input_dim=input_dim,
                    learning_rate=0.001,
                    dropout_rate=0.3,
                    layers=[128, 64, 32])

#Resumimos la estructura de la red
model.summary()

Tras definir la arquitectura del modelo, generaremos dos Callbacks que nos ayudarán a detener la época de entrenamiento cuando no hayan mejoras significativas durante un cierto periodo de tiempo **(con una paciencia de 10)** y a reducir la tasa de aprendizaje cuando el AUC de validación se estanque **(se reduce a la mitad cuando se estanca por 5 épocas)**, con el fin de reducir el tamaño del paso del optimizador y acelerar la convergencia.

In [ ]:
#6.3 --- Callbacks
early_stopping = EarlyStopping(
    monitor='val_AUC',
    patience=10,
    restore_best_weights=True,
    mode='max',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_AUC',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    mode='max',
    verbose=1
)

Después entrenamos el modelo durante **100 épocas**, con un tamaño de cada batch de **512 registros** y separando los datos en **80% para entrenamiento** y **20% para validación**, además balanceando las clases con los pesos calculados anteriormente y utilizando las callbacks definidas en el paso anterior.

In [ ]:
#6.4 --- Entrenamiento
history = model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=512,
    validation_split=0.2,
    class_weight=class_weight,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

Tras entrenar el modelo, evaluaremos los resultados, nuevamente mediante un reporte de clasificación, pero esta vez para la red neuronal y analizaremos el valor AUC-ROC.

In [ ]:
#6.5 --- Evaluación
y_pred_proba_nn = model.predict(X_test_scaled).flatten()
y_pred_nn = (y_pred_proba_nn >= 0.5).astype(int)

auc_nn = roc_auc_score(y_test, y_pred_proba_nn)

print("\n--- Reporte de clasificación (Red Neuronal) ---")
print(classification_report(y_test, y_pred_nn,
                             target_names=['Buen pagador', 'Mal pagador']))
print(f"AUC-ROC: {auc_nn:.4f}")



Tras evaluar nuestro modelo de red neuronal y analizar sus resultados, vemos que este sigue teniendo un accuracy del **65%** (en parte se debe a que ambos modelos están optimizados para balancear las clases, evitando calsificar a malos pagadores como buenos pagadores, sin importar si se escapan varios buenos pagadores clasificados como malos pagadores), además presenta un recall del **66%** para los malos pagadores. Por otra parte el valor AUC-ROC aumentó al **71.28%**, lo cual es modestamente mejor que el modelo implementado con regresión logística, pero no es el mejor modelo, puesto que todavía se está equivocando en clasificar casi el **30%** de los usuarios.

Para que lo anterior quede más claro, procedemos a graficar la curva de la función de pérdida, la curva AUC, junto con la matriz de confusión para ver en qué casos se está equivocando más la red, y finalmente compararemos las curvas ROC de ambos modelos (regresión logística y red neuronal).

In [ ]:
#6.6 --- Gráficos
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Gráfico 1 — Curva de pérdida
axes[0,0].plot(history.history['loss'], label='Train', color='#3498db')
axes[0,0].plot(history.history['val_loss'], label='Validación', color='#e74c3c')
axes[0,0].set_title('Figura 6a. Curva de pérdida', fontsize=12, fontweight='bold')
axes[0,0].set_xlabel('Época')
axes[0,0].set_ylabel('Loss (Binary Crossentropy)')
axes[0,0].legend()

# Gráfico 2 — Curva AUC
axes[0,1].plot(history.history['AUC'], label='Train', color='#3498db')
axes[0,1].plot(history.history['val_AUC'], label='Validación', color='#e74c3c')
axes[0,1].set_title('Figura 6b. Curva AUC durante entrenamiento', fontsize=12, fontweight='bold')
axes[0,1].set_xlabel('Época')
axes[0,1].set_ylabel('AUC-ROC')
axes[0,1].legend()

# Gráfico 3 — Matriz de confusión
cm_nn = confusion_matrix(y_test, y_pred_nn)
disp_nn = ConfusionMatrixDisplay(confusion_matrix=cm_nn,
                                  display_labels=['Buen pagador', 'Mal pagador'])
disp_nn.plot(ax=axes[1,0], colorbar=False, cmap='Blues')
axes[1,0].set_title('Figura 6c. Matriz de confusión Red Neuronal', fontsize=12, fontweight='bold')

# Gráfico 4 — Curva ROC comparativa
fpr_nn, tpr_nn, _ = roc_curve(y_test, y_pred_proba_nn)
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_pred_proba)

axes[1,1].plot(fpr_nn, tpr_nn, color='#3498db', lw=2, label=f'Red Neuronal (AUC = {auc_nn:.4f})')
axes[1,1].plot(fpr_lr, tpr_lr, color='#e74c3c', lw=2, label=f'Reg. Logística (AUC = {auc:.4f})')
axes[1,1].plot([0,1], [0,1], 'k--', lw=1, label='Clasificador aleatorio')
axes[1,1].fill_between(fpr_nn, tpr_nn, alpha=0.1, color='#3498db')
axes[1,1].set_xlabel('Tasa de Falsos Positivos')
axes[1,1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1,1].set_title('Figura 6d. Curva ROC comparativa', fontsize=12, fontweight='bold')
axes[1,1].legend(loc='lower right')

plt.suptitle('Figura 6. Evaluación de la Red Neuronal',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig6_red_neuronal.png', dpi=150, bbox_inches='tight')
plt.show()

A partir de las gráficas anteriores podemos concluir que el modelo de red neuronal supera ligeramente al modelo de regresión logística en todas las métricas, pero está muy lejos de ser el modelo ideal o por lo menos un modelo utilizable.

In [ ]:
#6.7 --- Comparación final
print("\n--- Comparación de modelos ---")
print(f"{'Modelo':<25} {'AUC-ROC':>10}")
print(f"{'-'*35}")
print(f"{'Regresión Logística':<25} {auc:>10.4f}")
print(f"{'Red Neuronal':<25} {auc_nn:>10.4f}")

Concluido esto, debemos de tomar otro tipo de medidas para mejorar el desempeño general del modelo.

## **7. Optimización de la red neuronal**

Para realizar la optimización de nuestra red comenzaremos utilizando la técnica de oversampling, es decir, balancearemos los datos para que tengan la misma cantidad de buenos y malos pagadores, en búsqueda de reducir afectaciones en la red o sesgos que se pudieron presentar por el peso calculado en pasos anteriores. Para hacer esto utilizaremos la clase SMOTE.

In [ ]:
#7 --- Optimización de la Red Neuronal

#7.1 --- SMOTE: Balanceo de clases por oversampling

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print("Distribución ANTES de SMOTE:")
print(pd.Series(y_train).value_counts())
print(f"\nDistribución DESPUÉS de SMOTE:")
print(pd.Series(y_train_sm).value_counts())
print(f"\nRegistros de entrenamiento antes: {len(y_train):,}")
print(f"Registros de entrenamiento después: {len(y_train_sm):,}")

Con esto obtenemos más registros para entrenar a nuestra red. Ahora buscaremos cual es la mejor arquitectura a implementar y los mejores hiperparámetros para el **dropout** y  **learning rate**, esto lo haremos utilizando diferentes configuraciones cada una tendrá su propia arquitectura de capas ocultas, neuronas y valores de hiperparámetros. Luego entrenaremos una red para cada combinación de estas, y hallaremos su valor AUC y su resultado para la métrica F1.

In [ ]:
#7.2 --- Búsqueda combinada de arquitectura e hiperparámetros

import itertools

arquitecturas = {
    'Muy profunda':  [512, 256, 128, 64, 32],
    'Embudo suave':  [128, 96, 64, 32],
    'Muy ancha':     [512, 256],
    'Ancha':         [256, 128],
    'Original':      [128, 64, 32],
}

dropouts = [0.3, 0.4, 0.5]
lrs      = [0.001, 0.0005]

combinaciones = list(itertools.product(arquitecturas.items(), dropouts, lrs))
print(f"Total combinaciones a probar: {len(combinaciones)}")

resultados_grid = []

for (nombre_arq, capas), dropout, lr in combinaciones:
    tf.random.set_seed(42)

    m = build_model(input_dim=input_dim,
                    learning_rate=lr,
                    dropout_rate=dropout,
                    layers=capas)

    m.fit(X_train_scaled, y_train,
          epochs=100,
          batch_size=512,
          validation_split=0.2,
          class_weight=class_weight,
          callbacks=[early_stopping, reduce_lr],
          verbose=0)

    proba    = m.predict(X_test_scaled).flatten()
    auc_temp = roc_auc_score(y_test, proba)
    f1_temp  = f1_score(y_test, (proba >= 0.5).astype(int), pos_label=1)

    resultados_grid.append({
        'Arquitectura':   nombre_arq,
        'Capas':          str(capas),
        'dropout':        dropout,
        'lr':             lr,
        'AUC-ROC':        round(auc_temp, 4),
        'F1 mal pagador': round(f1_temp, 4)
    })
    print(f"{nombre_arq} | dropout={dropout} | lr={lr}: AUC={auc_temp:.4f} | F1={f1_temp:.4f}")

df_grid = pd.DataFrame(resultados_grid).sort_values('AUC-ROC', ascending=False)
print("\nTop 10 combinaciones:")
print(df_grid.head(10).to_string(index=False))

mejor = df_grid.iloc[0]
mejor_arq      = arquitecturas[mejor['Arquitectura']]
mejor_dropout  = mejor['dropout']
mejor_lr       = mejor['lr']

print(f"\nMejor combinación:")
print(f"  Arquitectura: {mejor['Arquitectura']} {mejor_arq}")
print(f"  Dropout:      {mejor_dropout}")
print(f"  LR:           {mejor_lr}")
print(f"  AUC-ROC:      {mejor['AUC-ROC']}")
print(f"  F1:           {mejor['F1 mal pagador']}")

Encontramos que la mejor arquitectura es la **Ancha** con dos capas ocultas de **256** y **128** neuronas. Ahora buscaremos de la misma manera los mejores hiperparámetros para el **dropout** y la **tasa de aprendizaje**.

Como resultado obtenemos que los mejores hiperparámetros son **0.3** para el **dropout** y **0.001** para el **learning rate**. Con estos valores encontrados procedemos a elaborar el modelo final optimizado


In [ ]:
#7.3 --- Entrenamiento del modelo optimizado final

#Establecemos la semilla
tf.random.set_seed(42)

#Creamos el modelo
model_opt = build_model(input_dim=input_dim,
                        learning_rate=mejor_lr,
                        dropout_rate=mejor_dropout,
                        layers=mejor_arq)

#Generamos la arquitectura de este
print("Arquitectura del modelo optimizado:")
model_opt.summary()

#Incluímos las callbacks
early_stopping_opt = EarlyStopping(
    monitor='val_AUC',
    patience=15,
    restore_best_weights=True,
    mode='max',
    verbose=1
)

reduce_lr_opt = ReduceLROnPlateau(
    monitor='val_AUC',
    factor=0.5,
    patience=7,
    min_lr=1e-7,
    mode='max',
    verbose=1
)

#Entrenamos el modelo
history_opt = model_opt.fit(
    X_train_scaled, y_train,
    epochs=150,
    batch_size=1024,
    validation_split=0.2,
    class_weight=class_weight,
    callbacks=[early_stopping_opt, reduce_lr_opt],
    verbose=1
)

Tras entrenar el modelo optimizado, vamos a realizar otro ajuste implementando un umbral de clasificación óptimo, el cual le ayudará a nuestra red a decidir con confianza la clasificación de un individuo si la certeza en porcentaje de que este pertenezca a un grupo en específico es mayor que el umbral que buscamos optimizar en la siguiente celda de código.

In [ ]:
#7.4 --- Ajuste del umbral de clasificación óptimo

# Importar las métricas necesarias si no lo están ya
from sklearn.metrics import f1_score, recall_score, precision_score

#Obtenemos las predicciones del modelo
y_pred_proba_opt = y_pred_proba_nn.copy()

#Probaremos umbrales desde el 10% hasta el 90% con pasos de 1%
umbrales = np.arange(0.1, 0.9, 0.01)

#Almacenaremos las métricas para elegir el mejor umbral
f1_scores    = []
recall_scores = []
precision_scores = []

#Para cada umbral calculamos sus métricas y las guardamos
for u in umbrales:
    y_u = (y_pred_proba_opt >= u).astype(int)

    # F1 solo para mal pagador (pos_label=1)
    f1_scores.append(f1_score(y_test, y_u, pos_label=1, zero_division=0))

    # Recall solo para mal pagador
    recall_scores.append(recall_score(y_test, y_u, pos_label=1, zero_division=0))

    # Precision solo para mal pagador
    precision_scores.append(precision_score(y_test, y_u, pos_label=1, zero_division=0))

#Mostramos el mejor umbral
umbral_optimo = umbrales[np.argmax(f1_scores)]
print(f"Umbral óptimo (máximo F1): {umbral_optimo:.2f}")

#Gráfico de métricas vs umbral
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(umbrales, f1_scores,       label='F1 mal pagador',    color='#e74c3c', lw=2)
ax.plot(umbrales, recall_scores,   label='Recall mal pagador', color='#3498db', lw=2)
ax.plot(umbrales, precision_scores,label='Precision mal pagador', color='#2ecc71', lw=2)
ax.axvline(umbral_optimo, color='black', linestyle='--', lw=1.5,
           label=f'Umbral óptimo: {umbral_optimo:.2f}')
ax.set_xlabel('Umbral de clasificación')
ax.set_ylabel('Métrica')
ax.set_title('Figura 7a. Métricas vs Umbral de clasificación',
             fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig7a_umbral_optimo.png', dpi=150, bbox_inches='tight')
print()
plt.show()

Con el umbral optimizado anteriormente, encontramos que el valor óptimo para asignarle es **0.41**, es decir que el modelo tenderá a ser más conservador y clasificará a un cliente como mal pagador si este supera el **41%** de probabilidad de que sea mal pagador, en vez del **50%** que se asignaba por default, esto buscará reducir el riesgo de falsos positivos y buscará mejorar el desempeño general del modelo.

Ahora evaluaremos el desempeño del modelo optimizado y generaremos un reporte de clasificación para la Red Neuronal Optimizada.

In [ ]:
#7.5 --- Evaluación del modelo optimizado

y_pred_opt = (y_pred_proba_opt >= umbral_optimo).astype(int)
auc_opt    = roc_auc_score(y_test, y_pred_proba_opt)

print("--- Reporte de clasificación (Red Neuronal Optimizada) ---\n")
print(classification_report(y_test, y_pred_opt,
                             target_names=['Buen pagador', 'Mal pagador']))
print(f"AUC-ROC: {auc_opt:.4f}")
print(f"Umbral utilizado: {umbral_optimo:.2f}")

Observamos que en este caso el valor AUC-ROC es del **70.36%** lo cual es sorprendentemente menor que la red neuronal que desarrollamos en un principio, esto puede darse por muchas cosas, entre ellas el componente de aleatoriedad que quitamos al fijar la semilla para fortalecer la reproducibilidad del algoritmo que puedo haber estancado el entrenamiento de la red en un mínimo local o que pudo favorecer su overfitting, otra posible opción es que se haya alcanzado el techo predictivo del dataset, esto porque quizás influyen factores externos en la determinación del pago o no del crédito por parte de los clientes, y también porque se excluyeron variables consecuentes del incumplimiento o cumplimiento del pago del crédito, lo cual podría generar mayor accuracy pero no sería justo incluirlos en el análisis del modelo.

Nuevamente graficaremos la evaluación del modelo optimizado y su comparación con el modelo de regresión logística y la red neuronal anterior.

In [ ]:
#7.6 --- Gráficos de evaluación del modelo optimizado

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Gráfico 1 — Curva de pérdida
axes[0,0].plot(history_opt.history['loss'],     label='Train',     color='#3498db')
axes[0,0].plot(history_opt.history['val_loss'], label='Validación', color='#e74c3c')
axes[0,0].set_title('Figura 7b. Curva de pérdida', fontsize=12, fontweight='bold')
axes[0,0].set_xlabel('Época')
axes[0,0].set_ylabel('Loss (Binary Crossentropy)')
axes[0,0].legend()

# Gráfico 2 — Curva AUC
axes[0,1].plot(history_opt.history['AUC'],     label='Train',     color='#3498db')
axes[0,1].plot(history_opt.history['val_AUC'], label='Validación', color='#e74c3c')
axes[0,1].set_title('Figura 7c. Curva AUC durante entrenamiento', fontsize=12, fontweight='bold')
axes[0,1].set_xlabel('Época')
axes[0,1].set_ylabel('AUC-ROC')
axes[0,1].legend()

# Gráfico 3 — Matriz de confusión
cm_opt  = confusion_matrix(y_test, y_pred_opt)
disp_opt = ConfusionMatrixDisplay(confusion_matrix=cm_opt,
                                   display_labels=['Buen pagador', 'Mal pagador'])
disp_opt.plot(ax=axes[1,0], colorbar=False, cmap='Blues')
axes[1,0].set_title('Figura 7d. Matriz de confusión\nRed Neuronal Optimizada',
                    fontsize=12, fontweight='bold')

# Gráfico 4 — Curva ROC comparativa de los 3 modelos
fpr_opt, tpr_opt, _ = roc_curve(y_test, y_pred_proba_opt)
fpr_nn,  tpr_nn,  _ = roc_curve(y_test, y_pred_proba_nn)
fpr_lr,  tpr_lr,  _ = roc_curve(y_test, y_pred_proba)

axes[1,1].plot(fpr_opt, tpr_opt, color='#2ecc71', lw=2,
               label=f'Red Neuronal Optimizada (AUC = {auc_opt:.4f})')
axes[1,1].plot(fpr_nn,  tpr_nn,  color='#3498db', lw=2,
               label=f'Red Neuronal Base (AUC = {auc_nn:.4f})')
axes[1,1].plot(fpr_lr,  tpr_lr,  color='#e74c3c', lw=2,
               label=f'Reg. Logística (AUC = {auc:.4f})')
axes[1,1].plot([0,1], [0,1], 'k--', lw=1, label='Clasificador aleatorio')
axes[1,1].fill_between(fpr_opt, tpr_opt, alpha=0.1, color='#2ecc71')
axes[1,1].set_xlabel('Tasa de Falsos Positivos')
axes[1,1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1,1].set_title('Figura 7e. Curva ROC comparativa (3 modelos)',
                    fontsize=12, fontweight='bold')
axes[1,1].legend(loc='lower right', fontsize=9)

plt.suptitle('Figura 7. Evaluación de la Red Neuronal Optimizada',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig7_red_neuronal_optimizada.png', dpi=150, bbox_inches='tight')
plt.show()

Tras analizar las gráficas, se puede evidenciar que efectivamente hay overfitting en el modelo por lo que para el diseño final nos quedaremos con la red neuronal que implementamos en el apartado anterior, en búsqueda de mejorar el F1 score y el valor AUC-ROC.

In [ ]:
#7.7 --- Tabla comparativa final de los 3 modelos

print("--- Comparación final de modelos ---\n")
print(f"{'Modelo':<30} {'AUC-ROC':>10} {'Recall (malo)':>15} {'Precision (malo)':>18} {'F1 (malo)':>12}")
print("-" * 90)

from sklearn.metrics import precision_score, recall_score

# Regresión logística
p_lr  = precision_score(y_test, y_pred,    pos_label=1)
r_lr  = recall_score(y_test,    y_pred,    pos_label=1)
f1_lr = f1_score(y_test,        y_pred,    pos_label=1)

# Red neuronal base
p_nn  = precision_score(y_test, y_pred_nn, pos_label=1)
r_nn  = recall_score(y_test,    y_pred_nn, pos_label=1)
f1_nn = f1_score(y_test,        y_pred_nn, pos_label=1)

# Red neuronal optimizada
p_opt  = precision_score(y_test, y_pred_opt, pos_label=1)
r_opt  = recall_score(y_test,    y_pred_opt, pos_label=1)
f1_opt = f1_score(y_test,        y_pred_opt, pos_label=1)

print(f"{'Reg. Logística':<30} {auc:>10.4f} {r_lr:>15.4f} {p_lr:>18.4f} {f1_lr:>12.4f}")
print(f"{'Red Neuronal Base':<30} {auc_nn:>10.4f} {r_nn:>15.4f} {p_nn:>18.4f} {f1_nn:>12.4f}")
print(f"{'Red Neuronal Optimizada':<30} {auc_opt:>10.4f} {r_opt:>15.4f} {p_opt:>18.4f} {f1_opt:>12.4f}")

# Guardar el modelo optimizado
model_opt.save('modelo_riesgo_crediticio_optimizado.keras')
print("\nModelo optimizado guardado correctamente.")

La comparación de las métricas nos confirman nuestras sospechas y nos llevan a inclinarnos por el segundo modelo que fue el que nos dió mejores resultados.

## **8. Scorecard del modelo**

La Scorecard es la manera de traducir el resultado generado por el modelo en un puntaje numérico que cualquier persona pueda entender. Esto mismo es lo que usan bancos reales.



Para realizar la conversión de probabilidad de que sea un mal pagador a un puntaje permitido en una scorecard, nos apoyaremos en el estándar de la industria de créditos, utilizando un PDO (Points to Double the Odds) de **20** con un factor de **20/ln(2)**, un valor base de **600** puntos con **50%** de probabilidad de obtener odds 1:1.

In [ ]:
#8 --- Scorecard

#8.1 --- Conversión de probabilidad a puntaje

# Usamos la escala estándar de la industria crediticia
# PDO (Points to Double the Odds) = 20, Factor = 20/ln(2)
# Score base de 600 puntos con odds de 1:1 (50% probabilidad)

#Función para convertir la probabilidad a score
def probabilidad_a_score(probabilidad, pdo=20, score_base=600, odds_base=1):
    factor = pdo / np.log(2)
    offset = score_base - factor * np.log(odds_base)
    # Evitamos log(0) y log(1) con clip
    probabilidad = np.clip(probabilidad, 1e-6, 1 - 1e-6)
    odds = (1 - probabilidad) / probabilidad
    score = offset + factor * np.log(odds)
    # Limitamos el score al rango 300-850
    return np.clip(score, 300, 850)

# Calculamos el score para el conjunto de test
scores_test = probabilidad_a_score(y_pred_proba_nn)

#Mostramos los resultados de los scores para los datos de testing
print("Estadísticas del score en la población de test:")
print(f"  Score mínimo:  {scores_test.min():.1f}")
print(f"  Score máximo:  {scores_test.max():.1f}")
print(f"  Score medio:   {scores_test.mean():.1f}")
print(f"  Score mediana: {np.median(scores_test):.1f}")
print(f"\nScore promedio buenos pagadores: {scores_test[y_test == 0].mean():.1f}")
print(f"Score promedio malos pagadores:  {scores_test[y_test == 1].mean():.1f}")

De esta manera podemos determinar la confiabilidad de que un cliente sea buen o mal pagador utilizando un puntaje más amigable que simplemente un porcentaje. Ahora observemos la distribución del score en la población del testing.

In [ ]:
#8.2 --- Distribución del score en la población

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1 — Distribución general del score
axes[0].hist(scores_test[y_test == 0], bins=50, alpha=0.6,
             color='#2ecc71', label='Buen pagador', edgecolor='black', linewidth=0.3)
axes[0].hist(scores_test[y_test == 1], bins=50, alpha=0.6,
             color='#e74c3c', label='Mal pagador',  edgecolor='black', linewidth=0.3)
axes[0].axvline(scores_test.mean(), color='navy', linestyle='--', lw=1.5,
                label=f'Media general: {scores_test.mean():.0f}')
axes[0].set_xlabel('Score crediticio')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Figura 8a. Distribución del score por tipo de pagador',
                  fontsize=12, fontweight='bold')
axes[0].legend()

# Gráfico 2 — Boxplot del score por tipo de pagador
df_score = pd.DataFrame({'Score': scores_test,
                          'Tipo': y_test.map({0: 'Buen pagador',
                                              1: 'Mal pagador'}).values})
sns.boxplot(data=df_score, x='Tipo', y='Score', hue='Tipo',
            palette={'Buen pagador': '#2ecc71', 'Mal pagador': '#e74c3c'},
            legend=False, ax=axes[1])
axes[1].set_title('Figura 8b. Boxplot del score por tipo de pagador',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Score crediticio')

#Detalles generales del gráfico
plt.suptitle('Figura 8. Distribución del score crediticio',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig8_distribucion_score.png', dpi=150, bbox_inches='tight')
plt.show()

Podría generarse la necesidad de segmentar cada cliente según su nivel de riesgo para conocer que tanto riesgo hay de que sea un mal pagador. Esto lo exploramos en la siguiente celda de código.

In [ ]:
#8.3 --- Segmentación por nivel de riesgo

#Función para segmentar clientes según su score
def segmentar_riesgo(score):
    if score >= 750:
        return 'Muy bajo riesgo'
    elif score >= 650:
        return 'Bajo riesgo'
    elif score >= 550:
        return 'Riesgo medio'
    elif score >= 450:
        return 'Riesgo alto'
    else:
        return 'Muy alto riesgo'

#Utilizamos un Dataframe sobre los scores de la población del conjunto de testing
df_score['Segmento'] = df_score['Score'].apply(segmentar_riesgo)
df_score['target']   = y_test.values

#Establecemos el orden de los segmentos
orden_segmentos = ['Muy bajo riesgo', 'Bajo riesgo',
                   'Riesgo medio', 'Riesgo alto', 'Muy alto riesgo']

# Estadísticas por segmento
resumen = df_score.groupby('Segmento').agg(
    Clientes=('Score', 'count'),
    Score_medio=('Score', 'mean'),
    Tasa_incumplimiento=('target', 'mean')
).reindex(orden_segmentos)

resumen['Tasa_incumplimiento'] = (resumen['Tasa_incumplimiento'] * 100).round(1)
resumen['Score_medio'] = resumen['Score_medio'].round(1)
resumen['% Población'] = (resumen['Clientes'] / len(df_score) * 100).round(1)

#Generamos un resumen de la segmentación por nivel de riesgo
print("Tabla de segmentación por nivel de riesgo:")
print(resumen.to_string())

# Gráfico de tasa de incumplimiento por segmento
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colores_seg = ['#27ae60', '#2ecc71', '#f39c12', '#e67e22', '#e74c3c']

axes[0].bar(orden_segmentos, resumen['Tasa_incumplimiento'],
            color=colores_seg, edgecolor='black', linewidth=0.5)
axes[0].set_title('Figura 8c. Tasa de incumplimiento por segmento',
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('% Malos pagadores')
axes[0].tick_params(axis='x', rotation=30)
for i, val in enumerate(resumen['Tasa_incumplimiento']):
    axes[0].text(i, val + 0.3, f'{val:.1f}%', ha='center', fontweight='bold')

axes[1].bar(orden_segmentos, resumen['% Población'],
            color=colores_seg, edgecolor='black', linewidth=0.5)
axes[1].set_title('Figura 8d. Distribución de la población por segmento',
                  fontsize=12, fontweight='bold')
axes[1].set_ylabel('% de la población')
axes[1].tick_params(axis='x', rotation=30)
for i, val in enumerate(resumen['% Población']):
    axes[1].text(i, val + 0.3, f'{val:.1f}%', ha='center', fontweight='bold')

#Elementos generales del gráfico
plt.suptitle('Figura 8. Segmentación por nivel de riesgo',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig8_segmentacion_riesgo.png', dpi=150, bbox_inches='tight')
print()
plt.show()

Se percibe como se detectan a los malos pagadores con un alto riesgo, mientras que el riesgo medio es predominante en la pobleación general.

## **9. Variables más influyentes en el riesgo**

Finalmente analicemos cuales son aquellas variables que tienen una mayor contribución al Score, es decir aquellas que influencian más en la determinación de una decisión de parte del modelo para clasificar un determinado cliente como buen o mal pagador. Esto lo haremos calculando la contribución promedio por variable al score con el modelo de regresión logística que nos dió un AUC cercano al umbral del problema que encontramos en los modelos de redes neuronales, esto con el fin de simplificar el proceso.

In [ ]:
#8.4 --- Scorecard por variable (contribución al score)

# Calculamos la contribución promedio de cada variable al score
# usando los coeficientes de la regresión logística como proxy interpretable
coeficientes = pd.Series(logit.coef_[0], index=vars_disponibles)

# Top 15 variables con mayor impacto positivo y negativo
top_pos = coeficientes.nlargest(10)   # aumentan el riesgo
top_neg = coeficientes.nsmallest(10)  # reducen el riesgo
top_vars = pd.concat([top_neg, top_pos])

#Graficamos las variables en orden de impacto en el resultado final
fig, ax = plt.subplots(figsize=(12, 8))
colores_coef = ['#e74c3c' if v > 0 else '#2ecc71' for v in top_vars.values]
top_vars.plot(kind='barh', ax=ax, color=colores_coef, edgecolor='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Figura 8e. Variables que más aumentan o reducen el riesgo',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Coeficiente (positivo = más riesgo, negativo = menos riesgo)')
plt.tight_layout()
plt.savefig('fig8e_scorecard_variables.png', dpi=150, bbox_inches='tight')
plt.show()

#Mostramos las variables más y menos influyentes que encontramos anteriormente
print("\nTop 10 variables que MÁS AUMENTAN el riesgo:")
print(top_pos.round(4).to_string())
print("\nTop 10 variables que MÁS REDUCEN el riesgo:")
print(top_neg.round(4).to_string())

Podemos sacar la conclusión de que la variable más determinante dentro del modelo es el **int_rate** seguido del **term_60 months** y el **dti**, mientras que las variables menos influyentes o las que son más irrelevantes para la clasificación del modelo son el **annual_inc**, **total_acc** y **grade**. Estos resultados son similares a los que planteamos en nuestra hipótesis cuando hicimos la explroación inicial del dataset y de correlación entre variables.

## **10. Ejemplo de funcionamiento del modelo**

Así se vería el score asignado por el modelo a un nuevo cliente, pero en nuestro caso para no tener que diligenciar todos los valores de manera manual, tomaremos 3 ejemplos del conjunto de test.

In [ ]:
#8.5 --- Ejemplo de scoring para un cliente nuevo

#Tomamos 3 ejemplos reales del conjunto de test
ejemplos_idx = [0, 100, 500]

print("=" * 60)
print("EJEMPLOS DE SCORING PARA CLIENTES")
print("=" * 60)

#Calculamos los valores para cada caso
for idx in ejemplos_idx:
    cliente    = X_test.iloc[idx]
    realidad   = y_test.iloc[idx]
    prob       = y_pred_proba_nn[idx]
    score      = probabilidad_a_score(np.array([prob]))[0]
    segmento   = segmentar_riesgo(score)
    prediccion = 'Mal pagador' if prob >= 0.5 else 'Buen pagador'
    realidad_str = 'Mal pagador' if realidad == 1 else 'Buen pagador'

#Mostramos sus resultados
    print(f"\nCliente #{idx}")
    print(f"  Probabilidad de incumplimiento: {prob:.2%}")
    print(f"  Score crediticio:               {score:.0f} puntos")
    print(f"  Segmento de riesgo:             {segmento}")
    print(f"  Predicción del modelo:          {prediccion}")
    print(f"  Realidad:                       {realidad_str}")
    print(f"  ¿Acertó el modelo?              {'✓ Sí' if prediccion == realidad_str else '✗ No'}")
    print("-" * 60)

Finalmente guardamos nuestro modelo para que cuando requiramos utilizarlo no se necesite entrenar nuevamente, sino que ya esté listo y disponible para clasificar clientes que soliciten un crédito bancario.

In [ ]:
# Guardar
model.save('modelo_riesgo_crediticio.keras')

# Cargar en lugar de reentrenar
from tensorflow.keras.models import load_model
model = load_model('modelo_riesgo_crediticio.keras')